# Similarity Metrics Intuition (Cosine vs Dot vs L2)

**Purpose:** This is where “**Similarity ≠ correctness**” starts.

We will **compare three common metrics** used in embedding-based search:
- **Cosine similarity**
- **Dot product**
- **L2 (Euclidean) distance**

We’ll do it with **minimal math**, using a **controlled experiment**:
> Same embeddings, different metric → different ranking

By the end, you should be able to explain:
- what each metric is “really doing”
- why **normalization** matters
- why rankings change across metrics
- practical rules of thumb for text retrieval


## 0) Setup

We’ll use a fast sentence-transformer model and a tiny corpus.
If `sentence-transformers` isn’t installed, the first cell will install it.


In [2]:
# If needed, install sentence-transformers (safe to re-run)
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    !uv add sentence-transformers

In [3]:
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 120)

## 1) Three common metrics (no heavy math)

### A) Cosine similarity
- Measures the **angle** between vectors (direction)
- Ignores magnitude (length), *especially when vectors are normalized*

**Intuition:** “Do these vectors point in the same meaning direction?”

### B) Dot product
- Measures alignment **and** magnitude
- Larger magnitude vectors can score higher even if direction is similar

**Intuition:** “Direction + how strong/large is the vector?”

### C) L2 (Euclidean) distance
- Measures straight-line distance between points in space
- Smaller distance = more similar

**Intuition:** “How close are the points in meaning-space?”


## 2) Normalization: why it matters

**Normalization** means scaling vectors to have length 1.

When embeddings are normalized:
- **Cosine similarity** becomes equivalent to **dot product** (same ranking).
- **L2 distance** becomes tightly related to cosine (same ranking as well, just inverted).

When embeddings are *not* normalized:
- Dot product can behave very differently because it rewards large magnitudes.
- Cosine is often more stable because it focuses on direction.

We’ll show this empirically.


## 3) Load a tiny corpus + one query

We’ll use a corpus with multiple themes so rankings can change across metrics.


In [4]:
corpus = [
    "I love eating mangoes during summer.",
    "The chef prepared a spicy bowl of ramen.",
    "Coffee helps me stay focused during late-night work.",
    "I enjoy reading novels on rainy afternoons.",
    "The concert was loud, energetic, and unforgettable.",
    "Machine learning models can recognize patterns in data.",
    "We ran SQL queries to validate the dataset.",
    "We should refactor the code to reduce technical debt.",
    "Artificial intelligence is transforming many industries.",
    "He studied calculus to understand optimization.",
    "Banks assess credit risk before approving loans.",
    "The stock market fell sharply after the earnings report.",
    "The company launched a loyalty program to retain customers.",
    "She bought gasoline before driving to the province.",
    "Traffic on the expressway was heavy due to an accident.",
    "My dog chased a squirrel across the park.",
    "The athlete trained daily to improve endurance.",
    "A typhoon is expected to make landfall tomorrow evening.",
    "Photosynthesis converts sunlight into chemical energy.",
    "Insurance companies price policies based on risk exposure.",
]

query = "How do we reduce risk when giving loans to customers?"

len(corpus), query

(20, 'How do we reduce risk when giving loans to customers?')

## 4) Encode: embeddings (raw vs normalized)

We will compute:
- **raw embeddings** (not normalized)
- **normalized embeddings** (unit length)

Then we’ll compare rankings under different metrics.


In [5]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

# Raw (not normalized)
corpus_emb_raw = model.encode(corpus, normalize_embeddings=False)
query_emb_raw = model.encode([query], normalize_embeddings=False)[0]

# Normalized (unit vectors)
corpus_emb_norm = model.encode(corpus, normalize_embeddings=True)
query_emb_norm = model.encode([query], normalize_embeddings=True)[0]

print("Raw corpus embeddings:", corpus_emb_raw.shape)
print("Normalized corpus embeddings:", corpus_emb_norm.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Raw corpus embeddings: (20, 384)
Normalized corpus embeddings: (20, 384)


### Quick check: magnitudes differ in raw space

Dot product is sensitive to magnitude, so if raw vector lengths vary, it can change rankings.


In [6]:
raw_norms = np.linalg.norm(corpus_emb_raw, axis=1)

pd.DataFrame({
    "idx": np.arange(len(corpus)),
    "sentence": corpus,
    "raw_vector_norm": raw_norms
}).sort_values("raw_vector_norm", ascending=False).head(8)

,idx,sentence,raw_vector_norm
2,2,Coffee helps me stay focused during late-night work.,1.0
0,0,I love eating mangoes during summer.,1.0
6,6,We ran SQL queries to validate the dataset.,1.0
5,5,Machine learning models can recognize patterns in data.,1.0
18,18,Photosynthesis converts sunlight into chemical energy.,1.0
16,16,The athlete trained daily to improve endurance.,1.0
9,9,He studied calculus to understand optimization.,1.0
8,8,Artificial intelligence is transforming many industries.,1.0


## 5) Metric functions

We will compute scores for a given query against the corpus:

- **Cosine similarity**: higher is better
- **Dot product**: higher is better
- **L2 distance**: *lower* is better (we’ll rank by smallest distance)


In [7]:
def cosine_similarity_single(query_vec, matrix):
    # cos(q, x) = (q·x) / (||q|| ||x||)
    q = query_vec
    q_norm = np.linalg.norm(q) + 1e-12
    x_norm = np.linalg.norm(matrix, axis=1) + 1e-12
    return (matrix @ q) / (x_norm * q_norm)

def dot_product_single(query_vec, matrix):
    return matrix @ query_vec

def l2_distance_single(query_vec, matrix):
    return np.linalg.norm(matrix - query_vec, axis=1)

def top_k_rankings(query_vec, matrix, k=8, use_cosine=False, use_dot=False, use_l2=False):
    if use_cosine:
        scores = cosine_similarity_single(query_vec, matrix)
        order = np.argsort(-scores)
        metric_name = "cosine (higher=better)"
    elif use_dot:
        scores = dot_product_single(query_vec, matrix)
        order = np.argsort(-scores)
        metric_name = "dot (higher=better)"
    elif use_l2:
        scores = l2_distance_single(query_vec, matrix)
        order = np.argsort(scores)  # smaller distance is better
        metric_name = "L2 (lower=better)"
    else:
        raise ValueError("Choose a metric.")
    
    top_idx = order[:k]
    df = pd.DataFrame({
        "rank": np.arange(1, k+1),
        "idx": top_idx,
        "sentence": [corpus[i] for i in top_idx],
        "score": scores[top_idx],
    })
    df["metric"] = metric_name
    return df[["metric", "rank", "idx", "score", "sentence"]]

## 6) Controlled experiment: same embeddings, different metric → different ranking

We will compare rankings for the **same query** using **raw embeddings** first.


In [8]:
k = 8

raw_cos = top_k_rankings(query_emb_raw, corpus_emb_raw, k=k, use_cosine=True)
raw_dot = top_k_rankings(query_emb_raw, corpus_emb_raw, k=k, use_dot=True)
raw_l2  = top_k_rankings(query_emb_raw, corpus_emb_raw, k=k, use_l2=True)

pd.concat([raw_cos, raw_dot, raw_l2], ignore_index=True)

,metric,rank,idx,score,sentence
0,cosine (higher=better),1,10,0.545592,Banks assess credit risk before approving loans.
1,cosine (higher=better),2,7,0.328556,We should refactor the code to reduce technical debt.
2,cosine (higher=better),3,12,0.264296,The company launched a loyalty program to retain customers.
3,cosine (higher=better),4,19,0.253216,Insurance companies price policies based on risk exposure.
4,cosine (higher=better),5,13,0.121050,She bought gasoline before driving to the province.
5,cosine (higher=better),6,8,0.066078,Artificial intelligence is transforming many industries.
6,cosine (higher=better),7,5,0.058589,Machine learning models can recognize patterns in data.
7,cosine (higher=better),8,2,0.039874,Coffee helps me stay focused during late-night work.
8,dot (higher=better),1,10,0.545592,Banks assess credit risk before approving loans.
9,dot (higher=better),2,7,0.328556,We should refactor the code to reduce technical debt.


### Observation helper: how much did the ranking change?

We’ll compare the **top-k sets** and the **rank positions** for each metric.


In [9]:
def compare_rankings(df_a, df_b, name_a="A", name_b="B"):
    # df columns: idx, rank
    a = df_a[["idx", "rank"]].set_index("idx").rename(columns={"rank": f"rank_{name_a}"})
    b = df_b[["idx", "rank"]].set_index("idx").rename(columns={"rank": f"rank_{name_b}"})
    joined = a.join(b, how="outer")
    joined["in_both"] = joined.notna().all(axis=1)
    return joined.sort_values([f"rank_{name_a}"], na_position="last")

raw_change_cos_dot = compare_rankings(raw_cos, raw_dot, "cos", "dot")
raw_change_cos_l2  = compare_rankings(raw_cos, raw_l2, "cos", "l2")

raw_change_cos_dot.head(15), raw_change_cos_l2.head(15)

(     rank_cos  rank_dot  in_both
 idx                             
 10          1         1     True
 7           2         2     True
 12          3         3     True
 19          4         4     True
 13          5         5     True
 8           6         6     True
 5           7         7     True
 2           8         8     True,
      rank_cos  rank_l2  in_both
 idx                            
 10          1        1     True
 7           2        2     True
 12          3        3     True
 19          4        4     True
 13          5        5     True
 8           6        6     True
 5           7        7     True
 2           8        8     True)

## 7) Now repeat with normalized embeddings

When vectors are normalized:
- cosine ≈ dot (same ranking)
- L2 becomes strongly aligned with cosine

Let’s verify.


In [10]:
norm_cos = top_k_rankings(query_emb_norm, corpus_emb_norm, k=k, use_cosine=True)
norm_dot = top_k_rankings(query_emb_norm, corpus_emb_norm, k=k, use_dot=True)
norm_l2  = top_k_rankings(query_emb_norm, corpus_emb_norm, k=k, use_l2=True)

pd.concat([norm_cos, norm_dot, norm_l2], ignore_index=True)

,metric,rank,idx,score,sentence
0,cosine (higher=better),1,10,0.545592,Banks assess credit risk before approving loans.
1,cosine (higher=better),2,7,0.328556,We should refactor the code to reduce technical debt.
2,cosine (higher=better),3,12,0.264296,The company launched a loyalty program to retain customers.
3,cosine (higher=better),4,19,0.253216,Insurance companies price policies based on risk exposure.
4,cosine (higher=better),5,13,0.121050,She bought gasoline before driving to the province.
5,cosine (higher=better),6,8,0.066078,Artificial intelligence is transforming many industries.
6,cosine (higher=better),7,5,0.058589,Machine learning models can recognize patterns in data.
7,cosine (higher=better),8,2,0.039874,Coffee helps me stay focused during late-night work.
8,dot (higher=better),1,10,0.545592,Banks assess credit risk before approving loans.
9,dot (higher=better),2,7,0.328556,We should refactor the code to reduce technical debt.


In [11]:
norm_change_cos_dot = compare_rankings(norm_cos, norm_dot, "cos", "dot")
norm_change_cos_l2  = compare_rankings(norm_cos, norm_l2, "cos", "l2")

norm_change_cos_dot.head(15), norm_change_cos_l2.head(15)

(     rank_cos  rank_dot  in_both
 idx                             
 10          1         1     True
 7           2         2     True
 12          3         3     True
 19          4         4     True
 13          5         5     True
 8           6         6     True
 5           7         7     True
 2           8         8     True,
      rank_cos  rank_l2  in_both
 idx                            
 10          1        1     True
 7           2        2     True
 12          3        3     True
 19          4        4     True
 13          5        5     True
 8           6        6     True
 5           7        7     True
 2           8        8     True)

## 8) Interpretation: what changed and why?

### Why raw rankings change
- **Dot product** rewards magnitude. If a sentence embedding has a larger norm, it may rank higher even if direction is similar.
- **Cosine** focuses on direction, making it more robust across varying norms.
- **L2** compares absolute distances; it can behave differently if vectors are not normalized.

### Why normalization stabilizes things
When vectors are unit-length:
- dot(q, x) == cosine(q, x) (because ||q|| = ||x|| = 1)
- L2 distance is monotonic with cosine (so rankings line up)

So normalization often makes retrieval behavior more predictable.


## 9) Rules of thumb (practical)

- For **text retrieval** (semantic search), people often use:
  - **Cosine similarity** on normalized embeddings, or
  - **Dot product** on normalized embeddings (same effect, often faster)

- **Dot product without normalization** can be tricky:
  - It may over-rank vectors with large magnitude.

- **L2 distance** is fine when:
  - vectors are normalized, **or**
  - the model / index you use is designed around L2 (some ANN libraries / training objectives).

> **Big reminder:** High similarity does not mean “correct” — it only means the encoder thinks they are close in its meaning-space.


## Outputs checklist

- ✅ For a single query: top-k results under each metric (raw + normalized)
- ✅ “Ranking changed” observation (tables comparing positions)
- ✅ Interpretation + rules of thumb


## Mini reflection (student prompt)

1. Under **raw embeddings**, which result appears in the top-k for dot product but not for cosine? Why might that happen?
2. Under **normalized embeddings**, do cosine and dot match exactly for your top-k? If not, what could explain small differences?
3. If your retrieval system starts giving weird results, what’s the first thing you would check: **metric**, **normalization**, or **encoder choice**? Why?
